# Landscape Metrics: Plots

Reads the tidied/metric CSVs exported by the R Objective 3 pipeline (`scripts/r/`, specifically
`03_landscape_metrics.R`'s fragmentation/connectivity metrics and correlation screen) from
`outputs/tables/`.


In [ ]:
import geopandas as gpd
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import rasterio
import seaborn as sns
from matplotlib.colors import TwoSlopeNorm
from matplotlib.ticker import MaxNLocator

import config

In [ ]:
sns.set_theme(style="whitegrid")
config.PLOTS_DIR.mkdir(parents=True, exist_ok=True)

SITE_ORDER = [s["site_id"] for s in config.SITES]
SITE_LABELS = {s["site_id"]: s["site_name"] for s in config.SITES}

## Load tables

`landscape_connectivity_metrics_binary_natural_by_site_year_season.csv` is long-format (one row
per site/year/season-or-period/metric) period-composite rows (`baseline_2016_2018` etc.) carry
`year`/`season` as NA, seasonal per-year rows carry `period` as NA. Plots 1-5 below use the
seasonal rows only (`year` not null).
`landscape_metric_correlation_matrix.csv` is long-format pairwise correlations
(`Var1`/`Var2`/`Freq`) from `03_landscape_metrics.R`'s redundancy screen, computed across ALL
those seasonal + period observations pooled together.


In [ ]:
binary_metrics = pd.read_csv(config.TABLES_DIR / "landscape_connectivity_metrics_binary_natural_by_site_year_season.csv")
correlation_matrix = pd.read_csv(config.TABLES_DIR / "landscape_metric_correlation_matrix.csv")

## Plots 1-5: Natural-habitat fragmentation/connectivity metric trends by site

Five of the six headline metrics `03_landscape_metrics.R` computed on the binary natural-habitat
raster (classes 1-3 vs. 4-6), per site/year/season, 2016-2025 wet/dry seasonal composites. Color = site,
line style = season. PLAND is a percentage of the
*classified* (non-NA) area within each site, not of the site's full nominal polygon area. See
`landscape_valid_pixel_coverage_by_site_year_season.csv` for that ratio per site/year/season.


In [ ]:
def plot_landscape_metric_trend(metric: str, ylabel: str, filename: str) -> None:
    """Line plot of one binary natural-habitat landscape metric by site/season, 2016-2025."""
    sub = binary_metrics[(binary_metrics["metric"] == metric) & binary_metrics["year"].notna()].copy()
    sub["site_name"] = sub["site_id"].map(SITE_LABELS)
    sub["year"] = sub["year"].astype(int)

    fig, ax = plt.subplots(figsize=(12, 7))
    sns.lineplot(
        data=sub,
        x="year",
        y="value",
        hue="site_name",
        hue_order=[SITE_LABELS[s] for s in SITE_ORDER],
        style="season",
        markers=True,
        dashes=True,
        ax=ax,
    )
    ax.set_xlabel("Year")
    ax.set_ylabel(ylabel)
    ax.set_title(f"Natural-habitat {metric.upper()} by site")
    ax.legend(title=None, bbox_to_anchor=(1.02, 1), loc="upper left")
    fig.tight_layout()
    fig.savefig(config.PLOTS_DIR / filename, dpi=200, bbox_inches="tight")

### Plot 1: PLAND (% of classified area, natural)

In [ ]:
plot_landscape_metric_trend(
    "pland",
    "PLAND, % of classified area (natural)",
    "landscape_pland_by_site_trend.png",
)

### Plot 2: Patch density (PD)

In [ ]:
plot_landscape_metric_trend(
    "pd",
    "Patch density (patches / 100 ha)",
    "landscape_pd_by_site_trend.png",
)

### Plot 3: Largest patch index (LPI)

In [ ]:
plot_landscape_metric_trend(
    "lpi",
    "Largest patch index (%)",
    "landscape_lpi_by_site_trend.png",
)

### Plot 4: Cohesion

In [ ]:
plot_landscape_metric_trend(
    "cohesion",
    "Cohesion",
    "landscape_cohesion_by_site_trend.png",
)

### Plot 5: Effective mesh size (MESH)

In [ ]:
plot_landscape_metric_trend(
    "mesh",
    "Effective mesh size (ha)",
    "landscape_mesh_by_site_trend.png",
)

## Plot 6: Metric correlation matrix

Pairwise correlation (pooled across every site/year/season/period observation) among the nine
binary natural-habitat metrics (`ai`, `clumpy`, `cohesion`, `ed`, `enn_mn`, `lpi`, `mesh`, `pd`,
`pland`), from `03_landscape_metrics.R`'s redundancy screen -- the entropy pilot
(`landscape_entropy_pilot_by_site_year_season.csv`) is computed separately and not part of this
correlation matrix. A pair with `|r| > 0.85` that tells a similar ecological story should have
one metric dropped from the final six-headline-metric reporting set -- that's a human judgment
call for the report, not something this notebook decides automatically.

In [ ]:
corr_wide = correlation_matrix.pivot(index="Var1", columns="Var2", values="Freq")

fig, ax = plt.subplots(figsize=(8, 7))
sns.heatmap(
    corr_wide,
    annot=True,
    fmt=".2f",
    cmap="RdBu_r",
    center=0,
    vmin=-1,
    vmax=1,
    ax=ax,
    cbar_kws={"label": "r"},
)
ax.set_xlabel("")
ax.set_ylabel("")
ax.set_title("Landscape metric correlation matrix")
fig.tight_layout()
fig.savefig(config.PLOTS_DIR / "landscape_metric_correlation_heatmap.png", dpi=200, bbox_inches="tight")

## Plots 7-10: Pressure/threat context maps

Dynamic World conversion-pressure classification (`pressure_composition_by_period.png`
in `historical_change_plots.ipynb`) uses thresholds that were never calibrated against ground truth
(unlike `DW_HABITAT_THRESHOLDS`'s documented 3-round calibration) -- treat that chart as
preliminary. The four maps below are stronger, complementary evidence for the same "where is
pressure/threat coming from" question: `settlement_pressure_30m.tif`/`road_pressure_30m.tif`
(Objective 4, `06_prepare_connectivity_inputs.R`) are built from real GIS vector data (settlement
points, OSM road network), not an inferred spectral proxy, and `local_edge_density_change_*`/
`local_patch_density_change_*` (`04_moving_window_connectivity.R`) are raw, observed
baseline-to-current fragmentation change in real units, not a classification.

These are genuinely spatial (unlike Plots 1-6), but -- unlike the patch-graph/vector outputs noted
below, which need real topology exploration in QGIS -- each of these is a single-band continuous
raster that maps cleanly with a static `imshow` + colorbar, so a quick map belongs here too.


In [ ]:
site_boundaries = gpd.GeoDataFrame(
    pd.concat(
        [gpd.read_file(p).to_crs(config.PROJECT_CRS) for p in config.AOI_PATHS.values()],
        ignore_index=True,
    )
)


def plot_raster_map(raster_path, title, filename, cmap, cbar_label, diverging=False, vmin=None, vmax=None):
    """Static map of one single-band continuous raster, with site boundaries for context.

    diverging=True centers the colormap at 0 (TwoSlopeNorm) and, if vmin/vmax aren't given,
    picks a symmetric range from the 98th percentile of |value| -- robust to a few extreme
    per-window outliers rather than letting them wash out the rest of the map.
    """
    with rasterio.open(raster_path) as src:
        arr = src.read(1, masked=True).filled(np.nan)
        left, bottom, right, top = src.bounds

    norm = None
    if diverging:
        if vmax is None:
            vmax = np.nanpercentile(np.abs(arr), 98)
            vmin = -vmax
        norm = TwoSlopeNorm(vcenter=0, vmin=vmin, vmax=vmax)
        vmin = vmax = None  # norm supersedes vmin/vmax in imshow

    fig, ax = plt.subplots(figsize=(9, 8))
    im = ax.imshow(arr, extent=(left, right, bottom, top), origin="upper", cmap=cmap, norm=norm, vmin=vmin, vmax=vmax)
    site_boundaries.boundary.plot(ax=ax, color="black", linewidth=0.7)
    ax.set_title(title)
    ax.set_xlabel("Easting (m)", fontsize=10.5)
    ax.set_ylabel("Northing (m)", fontsize=10.5)

    # Full meter notation on both axes -- matplotlib defaults to a "x1e6"-style scientific
    # offset on the y axis once values get this large; disable it so y matches x's plain notation.
    ax.ticklabel_format(axis="both", style="plain", useOffset=False)
    # Slightly fewer gridlines than the default auto locator (was ~7 per axis at ~2000m spacing).
    ax.xaxis.set_major_locator(MaxNLocator(nbins=5))
    ax.yaxis.set_major_locator(MaxNLocator(nbins=5))
    ax.tick_params(axis="both", labelsize=9.5)
    ax.grid(True, alpha=0.5)  # same grid color as the notebook's whitegrid theme, just lighter

    cbar = fig.colorbar(im, ax=ax, shrink=0.8)
    cbar.set_label(cbar_label, fontsize=10.5)
    cbar.ax.tick_params(labelsize=9.5)

    fig.tight_layout()
    fig.savefig(config.PLOTS_DIR / filename, dpi=200, bbox_inches="tight")


### Plot 7: Settlement pressure (Objective 4)

In [ ]:
plot_raster_map(
    config.CONNECTIVITY_RASTER_DIR / "settlement_pressure_30m.tif",
    "Settlement Pressure (Kernel Density)",
    "connectivity_settlement_pressure.png",
    cmap="YlOrRd",
    cbar_label="Settlement pressure index (0-1)",
    vmin=0,
    vmax=1,
)

### Plot 8: Road pressure (Objective 4)

In [ ]:
plot_raster_map(
    config.CONNECTIVITY_RASTER_DIR / "road_pressure_30m.tif",
    "Road Pressure",
    "connectivity_road_pressure.png",
    cmap="YlOrRd",
    cbar_label="Road pressure index (0-1)",
    vmin=0,
    vmax=1,
)

### Plot 9: Local edge-density change, 500m window (Objective 3)

In [ ]:
# positive = more fragmented"
plot_raster_map(
    config.LANDSCAPE_RASTER_DIR / "local_edge_density_change_baseline_to_current_w500m.tif",
    "Local edge-density change, (500m window) baseline to current period",
    "landscape_edge_density_change_w500m_map.png",
    cmap="RdBu_r",
    cbar_label="Edge density change (m/ha)",
    diverging=True,
)

### Plot 10: Local patch-density change, 500m window (Objective 3)

In [ ]:
# positive = more fragmented"
plot_raster_map(
    config.LANDSCAPE_RASTER_DIR / "local_patch_density_change_baseline_to_current_w500m.tif",
    "Local patch-density change, 500m window (baseline -> current)",
    "landscape_patch_density_change_w500m_map.png",
    cmap="RdBu_r",
    cbar_label="Patch density change (patches/100ha)",
    diverging=True,
)

## Notes & next steps

- Patch-level and patch-graph outputs (`landscape_patch_metrics_current.csv`,
  `landscape_patch_importance_scores_current.csv`, `landscape_patch_graph_metrics_current.csv`,
  and the `outputs/vectors/*.gpkg` patch/graph/candidate-linkage-area files) are spatial/vector,
  not the kind of site/year trend or pairwise-correlation data this notebook charts -- view those
  in QGIS or similar rather than here.
- `landscape_metric_change_baseline_to_current.csv` / `..._pre_to_current.csv` (period-to-period
  deltas per site/metric) and `landscape_vs_objective2_crosscheck_by_site.csv` (the Mbokishi
  divergence caveat check) are tabular but small/summary in shape -- read directly rather than
  charted here; worth a simple bar chart in a future pass if the report needs one.
- Only seasonal (wet/dry per-year) rows are plotted in Plots 1-5; the same
  `landscape_connectivity_metrics_binary_natural_by_site_year_season.csv` also has period-composite
  rows (`year` is NA there) for baseline/pre/current period comparisons -- read those directly for
  period-level reporting rather than a per-year trend line.
